# glmtrust quickstart

Two variant effect predictors are rarely evaluated on the same variants, because each declines to
score a different part of the variant space. This notebook shows what that costs and what to do
about it, using the 11,109-variant fixture that ships with the package.

No raw data and no GPU. Everything here runs from a clone.

1. [Load the atlas scores](#1)
2. [The problem: coverage decides the comparison](#2)
3. [Calibrate and transfer across species](#3)
4. [Answer or abstain](#4)


In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")   # paths below are relative to the deposit root
import numpy as np
import polars as pl
import glmtrust

print("glmtrust", glmtrust.__version__)

glmtrust 0.1.1


<a id="1"></a>
## 1. Load the atlas scores

The fixture carries one row per variant: the species, an identifier, and the Evo 2 8,192-bp
mean-log-likelihood delta. Labels are encoded in the identifier, where a `neg_` prefix marks a
population variant and anything else is a curated pathogenic one.

In [2]:
from sklearn.metrics import roc_auc_score

fx = pl.read_parquet("glmtrust/benchmarks/fixtures/atlas8192_meanll.parquet").drop_nulls()

scores  = fx["evo2_meanll_delta"].to_numpy().astype(float)
labels  = np.array([0 if v.startswith("neg_") else 1 for v in fx["variant_id"]], dtype=int)
species = fx["species"].to_numpy()

# Orient each species so that a higher score means more deleterious. The stored delta's sign is a
# convention of the scorer, not of the biology, and leaving it unoriented reads the whole atlas
# upside down: AUROC 0.25 instead of 0.97. The benchmark script does exactly this.
for sp in np.unique(species):
    m = species == sp
    if len(np.unique(labels[m])) > 1 and roc_auc_score(labels[m], scores[m]) < 0.5:
        scores[m] = -scores[m]

print(f"{len(fx):,} variants across {len(set(species))} species")
print(f"{labels.sum():,} pathogenic, {(labels == 0).sum():,} population")

11,109 variants across 9 species
2,237 pathogenic, 8,872 population


<a id="2"></a>
## 2. The problem: coverage decides the comparison

A predictor that declines to score the variants it would have found hard reports a higher number
for the same skill. Below, a comparator scores every variant, and a second one refuses the hardest
20% of the positive class, which is the shape real conservation scores have.

Read on the variants each covers, the selective predictor looks better. Charged for what it cannot
answer, it does not.

In [3]:
rng = np.random.default_rng(0)

full = scores + rng.normal(0, 0.5, len(scores))          # answers everything
partial = full.copy()

# Refuse the positives this predictor scores least confidently: class-dependent by
# construction, which is the case that does not average away with more data.
pos = np.flatnonzero(labels == 1)
refused = pos[np.argsort(full[pos])[: int(0.20 * len(pos))]]
partial[refused] = np.nan

rep = glmtrust.audit(labels, [
    glmtrust.Scorer("answers everything",    full,    readout="simulated"),
    glmtrust.Scorer("declines the hard 20%", partial, readout="simulated"),
])
print(rep)

glmtrust audit
Intervals: variant-level. Where variants share a gene or locus they are too narrow; pass cluster= (CLI --cluster-col).

WARNINGS -- read these before the numbers
  ! answers everything: its values add nothing detectable beyond WHETHER it produced one. Ranking by that indicator and then by value gains +0.0066 AUROC over the indicator alone (95% CI [-0.0068, +0.0201], variant-level), so an accuracy quoted for it is carried by its reach.
  ! declines the hard 20% reaches 80.0% of positives against 100.0% of negatives (gap -0.200, 95% CI [-0.217, -0.184]). Its covered-subset accuracy is measured on an easier panel than the one it is reported for.
  ! declines the hard 20% cannot score 447 of 11109 variants; answering all of them at chance costs 0.0214 AUROC (0.6072 -> 0.5858).

REACH  (what each scorer can be run on at all)
  scorer         readout                  reach   on pos   on neg       gap
  answers everything simulated               100.0%   100.0%   100.0%   +0.00

The must-answer column is the one to compare. It charges every unscored variant the value a
coin flip would earn, which is the full-evaluation convention CAFA settled on for protein function
prediction and which variant effect prediction has not adopted.

In the paper, across 49 published human predictors on the ClinVar variants dbNSFP annotates, the
median must-answer penalty is 0.284, while the gene-level correction the field does make costs a
median 0.031 across the same predictors.

<a id="3"></a>
## 3. Calibrate and transfer across species

A raw likelihood delta is not a probability. `TrustLayer` fits a calibration map and a
class-conditional conformal layer, and `evaluate` scores each species with a map fitted on the
other eight, which is the situation a user in a label-poor species actually faces.

In [4]:
layer = glmtrust.TrustLayer(calibration="isotonic", conformal="mondrian",
                            alpha=0.10, coverage=0.85)
report = layer.fit(scores, labels).evaluate(scores, labels, groups=species)

d, c, k = report["discrimination"], report["calibration"], report["conformal"]
print(f"AUROC                {d['auroc']:.4f}  [{d['auroc_ci'][0]:.4f}, {d['auroc_ci'][1]:.4f}]")
print(f"expected calib error {c['ece']:.4f}")
print(f"Brier                {c['brier']:.4f}")
print()
print(f"conformal coverage   {k['coverage']:.4f}   at a nominal {1 - k['alpha']:.2f}")
print(f"  positive class     {k['coverage_pathogenic']:.4f}")
print(f"  benign class       {k['coverage_benign']:.4f}")
print(f"  abstention rate    {k['abstention_rate']:.4f}")

AUROC                0.9712  [0.9662, 0.9762]
expected calib error 0.0061
Brier                0.0320

conformal coverage   0.9092   at a nominal 0.90
  positive class     0.9307
  benign class       0.9037
  abstention rate    0.0270


<a id="4"></a>
## 4. Answer or abstain

The deliverable is not a better probability. Calibrated probabilities do not transfer across
species better than a two-parameter sigmoid. What transfers is the confidence *ordering*, and a
selective predictor needs only that: withhold each species' least-confident calls, and the errors
that remain are concentrated in what you kept.

These figures come from `evaluate`, which fits each species' calibration map on half of the other
species' variants and its conformal layer on the other half, so that the conformal quantile is fitted on
probabilities the map has not seen. The paper's trust-layer figures, a capture of 0.6096 and a lift of 4.07,
fit the map on all of them; `glmtrust/benchmarks/reproduce_paper_trust_layer.py` reproduces those from the
same fixture.

In [5]:
sel = report["selective"]

print(f"answering every variant   error rate {sel['full_error']:.4f}")
print(f"answering the top 85%     error rate {sel['selective_error']:.4f}")
print()
print(f"errors captured by the refused 15%   {sel['capture']:.4f}")
print(f"enrichment over refusing at random   {sel['lift']:.4f}x")

answering every variant   error rate 0.0376
answering the top 85%     error rate 0.0172

errors captured by the refused 15%   0.6124
enrichment over refusing at random   4.0829x


Removing more errors than calls is the whole point: the refused fraction is enriched for
mistakes, so the calls that survive are worth more than the average suggested.

One asymmetry decides how to describe this, and it is not the flattering one. Abstention suppresses
false alarms far better than it catches missed positives. It is a precision aid, not a safety
net, and a reader reaching for it as a screen should not.

### Where to go next

- `glmtrust/docs/tutorial.md` for the full API
- `glmtrust/benchmarks/reproduce_paper_trust_layer.py` to reproduce the paper's numbers exactly
- `docs/REPRODUCING.md` for what regenerates from this repository and what needs the raw data
